# Phase 7 -- Export data + model for the web dashboard

Produces everything `index.html` (the dashboard) looks for under `./data/`:

- `metadata.json` -- AOI bounds, band names, patch size, normalization stats, ONNX path
- `baseline_lst.json` -- predicted baseline LST as a flat grid
- `baseline_inputs.json` -- the 4 input bands as flat grids (needed for live re-inference)
- `uhvi.geojson` -- DUN polygons with a `uhvi` property (from your Phase 5 QGIS work)
- `telecom_sites.geojson` -- from Phase 6's `towers_df`
- `suitability.json` -- composite suitability raster (from Phase 6 section 7)
- `model.onnx` -- your trained U-Net, exported for browser inference

**Once these are generated**, put them in a `data/` folder next to `index.html` and deploy both
together (GitHub Pages, Netlify, Vercel all support static sites for free). No backend server
needed -- inference runs client-side via onnxruntime-web.

**Note on grid size:** the dashboard resamples/loads whatever resolution you export here directly
into the browser and into memory -- exporting the full-resolution Klang Valley raster (potentially
1800x1400+ pixels) as JSON would be large and slow to load. This notebook downsamples to a
web-friendly resolution (configurable, default ~150px on the long side) for the *display* layer,
while keeping the ONNX model itself running at full native patch resolution for live inference.

In [ ]:
import os
import json
import numpy as np
import rasterio
import torch

from google.colab import drive
drive.mount("/content/drive")

DRIVE_FOLDER = "UHI_Phase2_Exports"
export_dir = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
gis_ready_dir = f"{export_dir}/gis_ready"
telecom_dir = f"{export_dir}/telecom_risk"
checkpoint_path = f"{export_dir}/checkpoints/unet_lst_best.pt"

WEB_EXPORT_DIR = f"{export_dir}/web_dashboard_data"
os.makedirs(WEB_EXPORT_DIR, exist_ok=True)

DISPLAY_MAX_DIM = 150  # longest side of the downsampled display grid, in pixels

device = torch.device("cpu")  # ONNX export doesn't need GPU

## 1. Downsample and export the baseline LST raster

In [ ]:
def downsample(array, max_dim):
    h, w = array.shape
    scale = max_dim / max(h, w)
    new_h, new_w = max(1, int(h * scale)), max(1, int(w * scale))
    # simple block-mean downsample
    row_idx = np.linspace(0, h, new_h + 1).astype(int)
    col_idx = np.linspace(0, w, new_w + 1).astype(int)
    out = np.zeros((new_h, new_w), dtype=np.float32)
    for i in range(new_h):
        for j in range(new_w):
            block = array[row_idx[i]:row_idx[i+1], col_idx[j]:col_idx[j+1]]
            out[i, j] = np.nanmean(block) if block.size else np.nan
    return out


baseline_path = f"{gis_ready_dir}/industrial_park/baseline_lst.tif"
with rasterio.open(baseline_path) as src:
    baseline_full = src.read(1)
    bounds = src.bounds  # left, bottom, right, top -- lon/lat since TARGET_CRS = EPSG:4326

baseline_small = downsample(baseline_full, DISPLAY_MAX_DIM)
h, w = baseline_small.shape

with open(f"{WEB_EXPORT_DIR}/baseline_lst.json", "w") as f:
    json.dump({
        "width": w, "height": h,
        "bounds": [bounds.left, bounds.bottom, bounds.right, bounds.top],
        "values": np.nan_to_num(baseline_small, nan=np.nanmean(baseline_small)).round(2).tolist(),
    }, f)

print(f"Exported baseline_lst.json: {w}x{h} grid")

## 2. Export the 4 input bands at the same downsampled grid
These let the dashboard reconstruct a modified input for live inference. Ideally this reads the original band-separated tensor rather than re-deriving from the GeoTIFF stack; adjust `BAND_SOURCE_TIF` if your band layout differs.

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
INPUT_BANDS = checkpoint["input_bands"]  # e.g. ["NDVI", "NDBI", "ELEVATION", "ALBEDO"]
PATCH_SIZE = checkpoint["patch_size"]
input_mean = np.asarray(checkpoint["input_mean"]).tolist()
input_std = np.asarray(checkpoint["input_std"]).tolist()
target_mean = checkpoint["target_mean"]
target_std = checkpoint["target_std"]

# Reload the full-resolution tensor for the correct band order/values (BAND_NAMES from Phase 2)
BAND_NAMES = ["LST_C", "ALBEDO", "NDVI", "NDBI", "ELEVATION", "SLOPE"]
tensor = np.load(f"{export_dir}/training_tensor.npy")
baseline_t_index = tensor.shape[0] - 1  # same convention as Phase 4
band_idx = {b: BAND_NAMES.index(b) for b in INPUT_BANDS}

input_values = []
for b in INPUT_BANDS:
    full_band = tensor[baseline_t_index, :, :, band_idx[b]]
    small_band = downsample(full_band, DISPLAY_MAX_DIM)
    input_values.append(np.nan_to_num(small_band, nan=np.nanmean(small_band)).round(4).tolist())

with open(f"{WEB_EXPORT_DIR}/baseline_inputs.json", "w") as f:
    json.dump({"width": w, "height": h, "bands": INPUT_BANDS, "values": input_values}, f)

print("Exported baseline_inputs.json with bands:", INPUT_BANDS)

## 3. Export model metadata

In [ ]:
metadata = {
    "aoi_bounds": [bounds.left, bounds.bottom, bounds.right, bounds.top],
    "input_bands": INPUT_BANDS,
    "target_band": checkpoint["target_band"],
    "patch_size": PATCH_SIZE,
    "input_mean": input_mean,
    "input_std": input_std,
    "target_mean": target_mean,
    "target_std": target_std,
    "test_rmse": checkpoint.get("test_rmse"),
    "test_mae": checkpoint.get("test_mae"),
    "onnx_path": "./data/model.onnx",
}
with open(f"{WEB_EXPORT_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Exported metadata.json")
print(json.dumps(metadata, indent=2))

## 4. Export the trained U-Net to ONNX
This is what makes live, client-side inference possible with no backend server. Requires the same `UNet`/`ConvBlock` class definitions used in Phase 4 -- redefined here since a checkpoint only stores weights.

In [ ]:
import torch.nn as nn

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_channels, base_channels=32):
        super().__init__()
        c = base_channels
        self.enc1 = ConvBlock(in_channels, c)
        self.enc2 = ConvBlock(c, c * 2)
        self.enc3 = ConvBlock(c * 2, c * 4)
        self.enc4 = ConvBlock(c * 4, c * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(c * 8, c * 16)
        self.up4 = nn.ConvTranspose2d(c * 16, c * 8, 2, stride=2)
        self.dec4 = ConvBlock(c * 16, c * 8)
        self.up3 = nn.ConvTranspose2d(c * 8, c * 4, 2, stride=2)
        self.dec3 = ConvBlock(c * 8, c * 4)
        self.up2 = nn.ConvTranspose2d(c * 4, c * 2, 2, stride=2)
        self.dec2 = ConvBlock(c * 4, c * 2)
        self.up1 = nn.ConvTranspose2d(c * 2, c, 2, stride=2)
        self.dec1 = ConvBlock(c * 2, c)
        self.out_conv = nn.Conv2d(c, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.up4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out_conv(d1)


model = UNet(in_channels=checkpoint["in_channels"], base_channels=checkpoint["base_channels"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

dummy_input = torch.randn(1, checkpoint["in_channels"], PATCH_SIZE, PATCH_SIZE)
onnx_path = f"{WEB_EXPORT_DIR}/model.onnx"

torch.onnx.export(
    model, dummy_input, onnx_path,
    input_names=["input"], output_names=["output"],
    opset_version=18,
    dynamic_axes=None,  # fixed patch size -- matches the dashboard's tiling logic
)

print("Exported ONNX model to:", onnx_path)
print("Verify it loads correctly:")
import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("ONNX model check passed.")

## 5. Export UHVI (from your Phase 5 QGIS work) as GeoJSON
In QGIS: right-click your final UHVI layer -> Export -> Save Features As -> GeoJSON, make sure the `uhvi` field (or rename to match) is included, and CRS is EPSG:4326. Save it directly into `WEB_EXPORT_DIR` as `uhvi.geojson`, or upload it there via Drive. This cell just verifies the file looks right once you've done that.

In [ ]:
uhvi_path = f"{WEB_EXPORT_DIR}/uhvi.geojson"
if os.path.exists(uhvi_path):
    with open(uhvi_path) as f:
        uhvi_gj = json.load(f)
    print(f"Found uhvi.geojson with {len(uhvi_gj.get('features', []))} features")
    if uhvi_gj.get("features"):
        print("Sample properties:", uhvi_gj["features"][0]["properties"])
else:
    print(f"uhvi.geojson not found at {uhvi_path} -- export it from QGIS and place it here.")

## 6. Export telecom sites as GeoJSON (from Phase 6's `towers_df`)

In [ ]:
telecom_csv_path = f"{telecom_dir}/telecom_heat_risk.csv"
if os.path.exists(telecom_csv_path):
    import pandas as pd
    towers_df = pd.read_csv(telecom_csv_path)

    features = []
    for _, row in towers_df.iterrows():
        features.append({
            "type": "Feature",
            "properties": {
                "risk_tier": row["risk_tier"],
                "predicted_lst_c": round(float(row["predicted_lst_c"]), 2),
            },
            "geometry": {"type": "Point", "coordinates": [row["lon"], row["lat"]]},
        })

    telecom_geojson = {"type": "FeatureCollection", "features": features}
    with open(f"{WEB_EXPORT_DIR}/telecom_sites.geojson", "w") as f:
        json.dump(telecom_geojson, f)

    print(f"Exported telecom_sites.geojson with {len(features)} sites")
else:
    print(f"telecom_heat_risk.csv not found at {telecom_csv_path} -- run Phase 6 first.")

## 7. Export the composite suitability raster (from Phase 6 section 7)

In [ ]:
suitability_tif = f"{telecom_dir}/site_suitability_composite.tif"
if os.path.exists(suitability_tif):
    with rasterio.open(suitability_tif) as src:
        suit_full = src.read(1)
    suit_small = downsample(suit_full, DISPLAY_MAX_DIM)
    sh, sw = suit_small.shape

    with open(f"{WEB_EXPORT_DIR}/suitability.json", "w") as f:
        json.dump({
            "width": sw, "height": sh,
            "values": np.nan_to_num(suit_small, nan=0).round(4).tolist(),
        }, f)
    print(f"Exported suitability.json: {sw}x{sh} grid")
else:
    print(f"site_suitability_composite.tif not found at {suitability_tif} -- run Phase 6 section 7 first.")

## 8. Deploy

Download the whole `web_dashboard_data` folder from Drive, rename it to `data/`, and place it
next to `index.html`:

```
your-site/
  index.html
  data/
    metadata.json
    baseline_lst.json
    baseline_inputs.json
    uhvi.geojson
    telecom_sites.geojson
    suitability.json
    model.onnx
```

Then deploy the `your-site/` folder to any static host:
- **GitHub Pages** -- push to a repo, enable Pages on the branch, free
- **Netlify** -- drag-and-drop the folder onto app.netlify.com/drop, free, instant URL
- **Vercel** -- `vercel deploy` from the folder, free

Once deployed, open the URL -- the badge in the top-right corner should switch from
"DEMO DATA" to "REAL DATA" or "LIVE" automatically once it detects your exported files.